In [2]:
import pickle
import numpy as np
import networkx as nx
from src.plotting import *

from scipy.stats import t, ttest_rel

def extract_graphs(data_dict):
    nodes = np.array(data_dict["embedor_node_info"])
    edges = np.array(data_dict["embedor_edge_info"])
    embedor_emb = np.array(data_dict["embedor_emb"])
    embedor_apsp = np.array(data_dict["embedor_apsp"])
    # create networkx graph
    G = nx.Graph()
    G_low_energy = nx.Graph()

    for i, node in enumerate(nodes):
        G.add_node(i, emb=embedor_emb[i])
        G_low_energy.add_node(i, emb=embedor_emb[i])
    # add edges
    embedor_distances = []
    for edge in edges:
        G.add_edge(edge[0], edge[1])
        embedor_distances.append(embedor_apsp[edge[0], edge[1]])
    embedor_distances = np.array(embedor_distances)
    sorted_indices = np.argsort(embedor_distances)
    bottom_indices = sorted_indices[:int(len(sorted_indices)//3)]
    bottom_edges = [pair for i, pair in enumerate(edges) if i in bottom_indices]
    G_low_energy.add_edges_from(bottom_edges)
    return G, G_low_energy


def low_energy_edge_stats(embdng, full_graph, low_energy_graph, frac=1.0):
    # find average edge distance for original graph in embedding space
    distances = np.zeros(len(full_graph.edges()))
    for idx, (i, j) in enumerate(full_graph.edges()):
        dist = np.linalg.norm(embdng[i] - embdng[j])
        distances[idx] = dist
    # find the average distance
    avg_distance = np.mean(distances)
    # find the std of the distances
    std_distance = np.std(distances)

    # now compute z-scores for each low energy edge
    z_scores = np.zeros(len(low_energy_graph.edges()))
    for idx, (i, j) in enumerate(low_energy_graph.edges()):
        dist = np.linalg.norm(embdng[i] - embdng[j])
        z_scores[idx] = (dist - avg_distance) / std_distance
    z_scores_sorted = np.sort(z_scores)
    # return mean and std of top {100*pctg}% of z-scores
    top_z_scores = z_scores_sorted[-int(len(z_scores) * frac):]
    mean_z_score = np.mean(top_z_scores)
    std_z_score = np.std(top_z_scores)
    return mean_z_score, std_z_score, z_scores_sorted, z_scores

def low_distance_edge_stats(embdng, full_graph, apsp, frac=0.33):
    # find average edge distance for original graph in embedding space
    distances = np.zeros(len(full_graph.edges()))
    energy = np.zeros(len(full_graph.edges()))
    for idx, (i, j) in enumerate(full_graph.edges()):
        dist = np.linalg.norm(embdng[i] - embdng[j])
        distances[idx] = dist
        energy[idx] = apsp[i, j]
    z_scored_energies = (energy - np.mean(energy)) / np.std(energy)
    # take lowest frac*100% of edges with respect to energy
    sorted_indices = np.argsort(distances)
    bottom_indices = sorted_indices[:int(len(sorted_indices) * frac)]
    # get z scores of energies for these edges
    bottom_z_scores = z_scored_energies[bottom_indices]
    # return mean and std of top {100*pctg}% of z-scores
    mean_z_score = np.mean(bottom_z_scores)
    std_z_score = np.std(bottom_z_scores)
    return mean_z_score, std_z_score, bottom_z_scores


In [ ]:
## paired t-test - analyze z-scored distances of low energy edges

datasets = ['mnist', 'fmnist', 'developmental', 'macosko', 'chimp']

for dataset in datasets:
    print("*"*100)
    print(f"Processing dataset: {dataset}")
    dict_path = f"/home/tristan/Research/Sp25/embedor/outputs/server_experiments/embedor_{dataset}_25k.pkl"
    with open(dict_path, "rb") as f:
        data = pickle.load(f)

    G, G_low_energy = extract_graphs(data)
    _, _, _, z_scores = low_energy_edge_stats(data['embedor_emb'], G, G_low_energy)
    _, _, _, z_scores_umap = low_energy_edge_stats(data['umap_emb'], G, G_low_energy)
    _, _, _, z_scores_tsne = low_energy_edge_stats(data['tsne_emb'], G, G_low_energy)

    t_stat, p_val = ttest_rel(z_scores, z_scores_umap, alternative='less')

    print(f"Paired t-test (H₀: μ_emb = μ_umap, H₁: μ_emb < μ_umap): "
      f"t = {t_stat:.4f}, p = {p_val:.4g} — "
      f"{'REJECT' if p_val < 0.01 else 'FAIL TO REJECT'} H₀ at α = 0.01")
    
    t_stat, p_val = ttest_rel(z_scores, z_scores_tsne, alternative='less')
    print(f"Paired t-test (H₀: μ_emb = μ_tsne, H₁: μ_emb < μ_tsne): "
      f"t = {t_stat:.4f}, p = {p_val:.4g} — "
      f"{'REJECT' if p_val < 0.01 else 'FAIL TO REJECT'} H₀ at α = 0.01")
    print("*"*100)
    print()
    

****************************************************************************************************
Processing dataset: mnist


In [ ]:
## paired t-test: analyze z-scored energies of low-distance edges

datasets = ['mnist', 'fmnist', 'developmental', 'macosko', 'chimp']

for dataset in datasets:
    print("*"*100)
    print(f"Processing dataset: {dataset}")
    dict_path = f"/home/tristan/Research/Sp25/embedor/outputs/server_experiments/embedor_{dataset}_25k.pkl"
    with open(dict_path, "rb") as f:
        data = pickle.load(f)

    G, G_low_energy = extract_graphs(data)

    mean_z_score, std_z_score, bottom_z_scores = low_distance_edge_stats(data['embedor_emb'], G_low_energy, data['embedor_apsp'])
    print(f"Mean z-score of low energy edges: {mean_z_score:.4f}, std: {std_z_score:.4f}")
    mean_z_score_umap, std_z_score_umap, bottom_z_scores_umap = low_distance_edge_stats(data['umap_emb'], G_low_energy, data['embedor_apsp'])
    print(f"Mean z-score of low energy edges (UMAP): {mean_z_score_umap:.4f}, std: {std_z_score_umap:.4f}")
    mean_z_score_tsne, std_z_score_tsne, bottom_z_scores_tsne = low_distance_edge_stats(data['tsne_emb'], G_low_energy, data['embedor_apsp'])
    print(f"Mean z-score of low energy edges (t-SNE): {mean_z_score_tsne:.4f}, std: {std_z_score_tsne:.4f}")
    print()
    t_stat, p_val = ttest_rel(bottom_z_scores, bottom_z_scores_umap, alternative='less')
    print(f"Paired t-test (H₀: μ_emb = μ_umap, H₁: μ_emb < μ_umap): "
      f"t = {t_stat:.4f}, p = {p_val:.4g} — "
      f"{'REJECT' if p_val < 0.01 else 'FAIL TO REJECT'} H₀ at α = 0.01")
    
    t_stat, p_val = ttest_rel(bottom_z_scores, bottom_z_scores_tsne, alternative='less')
    print(f"Paired t-test (H₀: μ_emb = μ_tsne, H₁: μ_emb < μ_tsne): "
      f"t = {t_stat:.4f}, p = {p_val:.4g} — "
      f"{'REJECT' if p_val < 0.01 else 'FAIL TO REJECT'} H₀ at α = 0.01")

    print("*"*100)
    print()
    

****************************************************************************************************
Processing dataset: mnist
Mean z-score of low energy edges: -0.3308, std: 1.0207
Mean z-score of low energy edges (UMAP): -0.2591, std: 1.0222
Mean z-score of low energy edges (t-SNE): -0.3305, std: 1.0239

Paired t-test (H₀: μ_emb ≥ μ_umap, H₁: μ_emb < μ_umap): t = -8.6496, p = 2.709e-18 — REJECT H₀ at α = 0.01
Paired t-test (H₀: μ_emb ≥ μ_tsne, H₁: μ_emb < μ_tsne): t = -0.0435, p = 0.4827 — FAIL TO REJECT H₀ at α = 0.01
****************************************************************************************************

****************************************************************************************************
Processing dataset: fmnist
Mean z-score of low energy edges: -0.4414, std: 1.0306
Mean z-score of low energy edges (UMAP): -0.3898, std: 1.0394
Mean z-score of low energy edges (t-SNE): -0.4492, std: 1.0447

Paired t-test (H₀: μ_emb ≥ μ_umap, H₁: μ_emb < μ_umap): t = -6

In [3]:
# repeat with permutation test


datasets = ['mnist', 'fmnist', 'developmental', 'macosko', 'chimp']

for dataset in datasets:
    print("*"*100)
    print(f"Processing dataset: {dataset}")
    dict_path = f"/home/tristan/Research/Sp25/embedor/outputs/server_experiments/embedor_{dataset}_25k.pkl"
    with open(dict_path, "rb") as f:
        data = pickle.load(f)

    G, G_low_energy = extract_graphs(data)
    _, _, _, z_scores = low_energy_edge_stats(data['embedor_emb'], G, G_low_energy) # gives bottom z-scores
    _, _, _, z_scores_umap = low_energy_edge_stats(data['umap_emb'], G, G_low_energy)
    _, _, _, z_scores_tsne = low_energy_edge_stats(data['tsne_emb'], G, G_low_energy)
    del G, G_low_energy
    # Perform permutation test
    from scipy.stats import permutation_test
    result = permutation_test((z_scores, z_scores_umap),
                              statistic=lambda x, y: np.mean(x) - np.mean(y),
                              batch=1000,
                              alternative='less',
                              n_resamples=10000,
                              random_state=42)
    print(f"Permutation test result UMAP: p-value = {result.pvalue:.4g}, "
          f"{'REJECT' if result.pvalue < 0.01 else 'FAIL TO REJECT'} H₀ at α = 0.01")
    result = permutation_test((z_scores, z_scores_tsne),
                              statistic=lambda x, y: np.mean(x) - np.mean(y),
                              batch=1000,
                              alternative='less',
                              n_resamples=10000,
                              random_state=42)
    print(f"Permutation test result tSNE: p-value = {result.pvalue:.4g}, "
          f"{'REJECT' if result.pvalue < 0.01 else 'FAIL TO REJECT'} H₀ at α = 0.01")
    print("*"*100)
    
    print()
    

****************************************************************************************************
Processing dataset: mnist
Permutation test result UMAP: p-value = 9.999e-05, REJECT H₀ at α = 0.01
Permutation test result tSNE: p-value = 0.9593, FAIL TO REJECT H₀ at α = 0.01
****************************************************************************************************

****************************************************************************************************
Processing dataset: fmnist
Permutation test result UMAP: p-value = 9.999e-05, REJECT H₀ at α = 0.01
Permutation test result tSNE: p-value = 9.999e-05, REJECT H₀ at α = 0.01
****************************************************************************************************

****************************************************************************************************
Processing dataset: developmental
Permutation test result UMAP: p-value = 9.999e-05, REJECT H₀ at α = 0.01
Permutation test result tSNE: p-valu